# DeepSeek-Prover-V2-7B as a remote prover for GraphConjecturing

Run this on a **Colab GPU runtime** (`Runtime → Change runtime type → GPU`; a free T4 works for the 7B model). It serves DeepSeek-Prover-V2 over HTTP and prints a public URL you paste into the pipeline's `CONFIG.prover_api_url`.

The pipeline re-compiles every returned proof against its own pinned mathlib and only counts kernel-verified ones — so this notebook just has to generate candidates.

**The big model (671B) does not fit on Colab.** Use the 7B here; point at an A100/H100 host for the 671B.

In [ ]:
# 1. Check the GPU
!nvidia-smi

In [ ]:
# 2. Install serving deps (vLLM pulls a matching torch). ~3-5 min.
!pip install -q vllm fastapi uvicorn nest_asyncio pyngrok transformers

In [ ]:
# 3. Grab the server module from the repo (or upload deepseek_prover_server.py yourself).
#    Easiest: paste the file contents into a cell, or fetch from your fork:
# !wget -q https://raw.githubusercontent.com/<you>/GraphConjecturing/main/colab/deepseek_prover_server.py
import os
assert os.path.exists('deepseek_prover_server.py'), 'upload deepseek_prover_server.py into this Colab session first'

In [ ]:
# 4. Start the FastAPI server in the background (loads the model on first request).
import subprocess, time, sys
server = subprocess.Popen([sys.executable, 'deepseek_prover_server.py'],
                          env={**os.environ, 'PORT': '8000'})
time.sleep(5)
print('server pid', server.pid)

In [ ]:
# 5a. Expose it with a free Cloudflare quick tunnel (no signup).
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, re, time
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    print(line, end='')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n\n>>> PUBLIC URL:', url)
print('>>> On the pipeline machine set:')
print(f'      CONFIG.prover_api_url = "{url}"')
print( '      CONFIG.prover_backends = ("lean", "deepseek")')

In [ ]:
# 5b. (alternative) ngrok — needs a free authtoken from dashboard.ngrok.com
# from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_TOKEN')
# print('PUBLIC URL:', ngrok.connect(8000).public_url)

In [ ]:
# 6. Smoke-test the endpoint (first call downloads + loads the model: a few min).
import requests
r = requests.post(url + '/v1/lean4/prove', json={
    'statement': 'theorem t (n : Nat) : n + 0 = n := by sorry',
    'informal_statement': 'n + 0 = n', 'timeout_s': 180}, timeout=600)
print(r.json())